Attention ces vérifs ne peuvent être réalisées qu'une fois les df finaux téléchanrgés et les defs de vérification importées

Import des fichiers .PY

In [ ]:
sys.path.append(os.path.abspath("/content/Code-PAT/scripts_data"))
sys.path.append(os.path.abspath("/content/Code-PAT"))

from pegass2 import *
from utils import *
from verif_maraude import *
from Maraude2 import *

#Vérification de Maraude

In [ ]:
verifier_colonne_structure(df_Nb_maraudes_SIGMA_struct, "n_structure", df_ref_structure)
verifier_colonne_structure(df_Nb_maraudes_SIGMA_DT, "DT_de_rattachement", df_ref_structure)
verifier_colonne_structure(df_nb_contacts_SIGMA_struct, "n_structure", df_ref_structure)
verifier_colonne_structure( df_nb_contacts_SIGMA_DT, "DT_de_rattachement", df_ref_structure)
verifier_colonne_structure(df_nb_personnes_rencontrees_SIGMA_struct, "n_structure", df_ref_structure)
verifier_colonne_structure(df_nb_personnes_rencontrees_SIGMA_DT, "DT_de_rattachement", df_ref_structure)


In [ ]:
df_maraude_verif= prep_nb_personnes_rencontrees_sigma(df_maraude, filtre_annee_fn=None)

df_maraude_verif= add_nb_personnes(df_maraude_verif,
                     out_col="nb personnes",
                     cols=None,
                     col_typologie="maraude_rencontre_typologie")

In [ ]:
run_verif_maraude(
    df_Nb_maraudes_SIGMA_struct=df_Nb_maraudes_SIGMA_struct,
    df_Nb_maraudes_SIGMA_DT=df_Nb_maraudes_SIGMA_DT,
    df_nb_contacts_SIGMA_struct=df_nb_contacts_SIGMA_struct,
    df_nb_contacts_SIGMA_DT=df_nb_contacts_SIGMA_DT,
    df_nb_personnes_rencontrees_SIGMA_struct=df_nb_personnes_rencontrees_SIGMA_struct,
    df_nb_personnes_rencontrees_SIGMA_DT=df_nb_personnes_rencontrees_SIGMA_DT,
    df_Nb_maraudes_SIGMA=df_Nb_maraudes_SIGMA,
    df_maraude_verif=df_maraude_verif,
    df_personnes_diff_base=df_personnes_diff_base,
)


Vérification de PEGASS

In [ ]:

# Vérif cohérence nb bénévoles :
# On contrôle que, pour un périmètre d’activités (ex : Maraude/AEO/IS),
# le TOTAL des couples uniques (STRUCTURE, NIVOL) dans la table source
# = la SOMME du nb de bénévoles (par structure) dans la table finale.
# -> Si égal : pas de perte / doublon dans le calcul de l’indicateur.

check_sum_vs_unique_pairs(
    nb_ben_Maraude_Pegass,
    "Maraude Nb_benevoles_actifs",
    df_pegass_ben_activite_synthetique,
    Activite_maraude,
    label="Maraude"
)

check_sum_vs_unique_pairs(
    nb_ben_AEO_Pegass,
    "AEO Nb_benevoles_actifs",
    df_pegass_ben_activite_synthetique,
    AEO,
    label="AEO/AAD"
)

check_sum_vs_unique_pairs(
    nb_ben_IS_Pegass,
    "IS Nb_benevoles_actifs",
    df_pegass_ben_activite_synthetique,
    IS_actifs,
    label="IS actifs"
)

In [ ]:

#Maraude
#Préparation de la table de vérif pour Maruade
df_maraude_verif = df_pegass_activite_merge2[df_pegass_activite_merge2["ACTIVITE_BENEVOLE_ID_FK"].isin(Activite_maraude)].copy()
#on enlève l'ensemble des activités n'ayant pas de structure d'appartenance
df_maraude_verif = df_maraude_verif[df_maraude_verif["PEGASS_ACTIVITE_STRUCTURE_MENANT_ACTIVITE_ID_FK"].notna()].copy()
#On enlève les maraudes sans structure présente dans le ref
df_maraude_verif = pd.merge(df_maraude_verif, ref_structure1, left_on = "PEGASS_ACTIVITE_STRUCTURE_MENANT_ACTIVITE_ID_FK", right_on= "n_structure", how='inner')
check_len_df_equals_sum_other(df_maraude_verif , nb_Maraude_Pegass1, "nb_Maraude_Pegass", label="Maraude")

# --- OPERATIONS ---
df_operations_verif = df_pegass_activite_merge2[df_pegass_activite_merge2["ACTIVITE_BENEVOLE_ID_FK"].isin(NB_operations)].copy()
df_operations_verif = df_operations_verif[df_operations_verif["PEGASS_ACTIVITE_STRUCTURE_MENANT_ACTIVITE_ID_FK"].notna()].copy()
check_len_df_equals_sum_other(df_operations_verif, nb_operations_Pegass1, "Dispositifs_d_urgence Nb_operations", label="Opérations")

# --- EXERCICES ---
df_exercices_verif = df_pegass_activite_merge2[df_pegass_activite_merge2["ACTIVITE_BENEVOLE_ID_FK"].isin(Nb_Exercice)].copy()
df_exercices_verif = df_exercices_verif[df_exercices_verif["PEGASS_ACTIVITE_STRUCTURE_MENANT_ACTIVITE_ID_FK"].notna()].copy()
check_len_df_equals_sum_other(df_exercices_verif, nb_Exercice_Pegass1, "Dispositifs_d_urgence Nb_exercices", label="Exercices")

# --- AEO / AAD ---
df_aeo_verif = df_pegass_activite_merge2[df_pegass_activite_merge2["ACTIVITE_BENEVOLE_ID_FK"].isin(AEO)].copy()
df_aeo_verif = df_aeo_verif[df_aeo_verif["PEGASS_ACTIVITE_STRUCTURE_MENANT_ACTIVITE_ID_FK"].notna()].copy()
check_len_df_equals_sum_other(df_aeo_verif, nb_AEO_Pegass1, "nb_activite_AEO", label="AEO/AAD")


In [ ]:
check_sum_same_col(nb_Maraude_Pegass1, Indics_pegass_struct, col="nb_Maraude_Pegass")
check_sum_same_col(nb_operations_Pegass1, Indics_pegass_struct, col="Dispositifs_d_urgence Nb_operations")
check_sum_same_col(nb_Exercice_Pegass1, Indics_pegass_struct, col="Dispositifs_d_urgence Nb_exercices")

In [ ]:
check_sum_same_col(nb_Maraude_Pegass1, Indics_pegass_DT, col="nb_Maraude_Pegass")
check_sum_same_col(nb_operations_Pegass1, Indics_pegass_DT, col="Dispositifs_d_urgence Nb_operations")
check_sum_same_col(nb_Exercice_Pegass1, Indics_pegass_DT, col="Dispositifs_d_urgence Nb_exercices")

In [ ]:
#VERIF au global
verifier_colonne_structure(Indics_pegass_struct,"n_structure", ref_structure1)